### Part 1: Cohort Analysis & Retention Analytics

##### Question 1 (First-Time vs Repeat Customers): 
Identify customer purchasing behavior by calculating the total count of customers who have placed exactly 1 order versus those who placed 2 or more orders in olist_orders_dataset.

In [0]:
with customer_total_orders as (
select customer_unique_id,count(order_id)  total_order
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord 
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on 
ord.customer_id=cus.customer_id
group by customer_unique_id)
select 
case
when total_order = 1 then '1 Order'
else '2 or more'
end as customer_type,
count(customer_unique_id) as total_customer
from customer_total_orders
group by customer_type;

In [0]:
with customer_total_order as (
    select customer_unique_id,count(order_id)  total_order
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord 
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on 
ord.customer_id=cus.customer_id 
group by customer_unique_id)
select count(case when total_order = 1 then 1 end) as single_order_customer,
count(case when total_order > 1 then 1 end) as multiple_order_customer
from customer_total_order;

##### Question 2 (Customer Cohort Month): 
For each customer (customer_unique_id via joining olist_customers_dataset), determine their cohort month (their very first purchase month). Group customers by cohort month and track total unique customers per cohort.

In [0]:
with customer_cohort as (
select cus.customer_unique_id,date_trunc('month',min(ord.order_purchase_timestamp)) cohort_month
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on 
ord.customer_id=cus.customer_id
group by cus.customer_unique_id)
select cohort_month,count(customer_unique_id) as total_customer
from customer_cohort
group by cohort_month
order by total_customer desc;

##### Question 3 (Customer Lifetime Value - LTV Quartiles):

Calculate the total lifetime spend per customer_unique_id (summing payment_value from olist_order_payments_dataset across their orders), and rank customers into spend quartiles using NTILE(4).

In [0]:
select customer_unique_id,sum(payment_value) as total_spend,
ntile(4) over(order by sum(payment_value) desc) as spend_quartile
from brazilian_e_commerce.sql_practice.olist_customers_dataset cus
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on cus.customer_id=ord.customer_id
join brazilian_e_commerce.sql_practice.olist_order_payments_dataset pay on ord.order_id=pay.order_id
group by customer_unique_id
order by total_spend desc;

### Part 2: Funnel, Performance & Lead Time Engineering

##### Question 4 (SLA / Late Delivery Analysis): 
Identify orders where order_delivered_customer_date exceeded order_estimated_delivery_date. Calculate the total count and percentage of late orders across the entire platform.

In [0]:
select count(order_id) tot_count_late_ord,
round(
    (count(order_id)*100.0)/(select count(order_id) from brazilian_e_commerce.sql_practice.olist_orders_dataset) ,2) as pct_late_ord
from brazilian_e_commerce.sql_practice.olist_orders_dataset
where order_delivered_customer_date > order_estimated_delivery_date;

##### Question 5 (Fulfillment Bottleneck Analysis): Break down the total fulfillment duration into two stages for delivered orders:

1.Approval delay: DATEDIFF(order_approved_at, order_purchase_timestamp)

2.Carrier shipping delay: DATEDIFF(order_delivered_customer_date, order_delivered_carrier_date)
Return the average days for both stages per year-month.

In [0]:
select date_trunc('month',order_purchase_timestamp) as month,
round(avg(date_diff(order_approved_at,order_purchase_timestamp)),2) approval_delay,
round(avg(date_diff(order_delivered_customer_date,order_delivered_carrier_date)),2) carrier_delay
from brazilian_e_commerce.sql_practice.olist_orders_dataset 
where order_status = 'delivered'
group by date_trunc('month',order_purchase_timestamp)
order by month;

##### Question 6 (Order Status Funnel Drop-off):

Calculate the absolute count of orders and the percentage breakdown for each order_status across the entire olist_orders_dataset. Order the results by total orders descending.

In [0]:
select order_status,count(order_id) order_count,
round(count(order_id)*100.0/(select count(order_id) from brazilian_e_commerce.sql_practice.olist_orders_dataset),2) pct_order
from brazilian_e_commerce.sql_practice.olist_orders_dataset
group by order_status
order by order_count desc;